# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library. All dataset entities are referenced by their `@id` for full reproducibility and auditability.

### Dataset Source
The dataset source is provided via this Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed in your environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# View basic metadata
print("Dataset Title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Version:", dataset.metadata.version)
print("Published:", dataset.metadata.datePublished)


## 2. Data Overview
Review available record sets and their field and column `@id`s. Below we print:
- The list of available Record Sets (with their `@id` and name).
- Their fields (and corresponding `@id`s) and columns.

This provides a map of all data tables inside the FAIR<sup>2</sup> dataset.

In [ ]:
# List all record sets, fields, and columns by their @id
def print_overview(ds):
    rsets = list(ds.record_sets())
    print(f"Found {len(rsets)} record sets.")
    for rset in rsets:
        print(f"\nRecord set: {rset['@id']}  (name: {rset.get('name', '')})")
        # Print associated fields
        fields = rset.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        if len(fields):
            print(" Fields:")
            if isinstance(fields, list):
                for f in fields:
                    if isinstance(f, dict):
                        print(f"   - {f['@id']} ({f.get('name','')})")
                    else:
                        print(f"   - {f}")
        # Print associated columns (if any)
        cols = rset.get('column', [])
        if isinstance(cols, dict):
            cols = [cols]
        if len(cols):
            print(" Columns:")
            for c in cols:
                if isinstance(c, dict):
                    print(f"   - {c['@id']} ({c.get('name', '')})")
                else:
                    print(f"   - {c}")

print_overview(dataset)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. You can extract all record sets or select one of interest by its `@id`.

In [ ]:
# Collect all available record set @id's
record_set_ids = [rset['@id'] for rset in dataset.record_sets()]
print('Record Set @id List:')
print(record_set_ids)

# Extract data from each record set (using only the first for demo)
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}.")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# Choose the main table for analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id is not None:
    print(f'Columns in main record set ({main_record_set_id}):')
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply standard data processing to one numeric field referenced by its `@id` (from the previous section's columns). Example EDA: Filtering records with a threshold, normalization of a numeric column, then grouping by another field (e.g., a protein or sample group) if present.

In [ ]:
# Choose a numeric field for demonstration, based on the record set columns
import numpy as np

# For this dataset, let's try to use the field columns inferred from main_record_set_id
df = dataframes[main_record_set_id]

possible_numeric_fields = [c for c in df.columns if df[c].dtype in [np.float64, np.int64] or pd.api.types.is_numeric_dtype(df[c])]
print('Identified numeric fields:', possible_numeric_fields)

# Select first numeric field available, fallback to 'coverage_percentage' if known
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
elif 'coverage_percentage' in df.columns:
    numeric_field_id = 'coverage_percentage'  # Adjust to your actual field @id or column name
else:
    raise Exception('No numeric field available for EDA!')

threshold = df[numeric_field_id].median() if numeric_field_id in df.columns else None
filtered_df = df[df[numeric_field_id] > threshold].copy() if threshold is not None else df.copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field, e.g., 'sample_id' if present, else use any string column
group_fields = [c for c in df.columns if c != numeric_field_id and df[c].dtype == object]
group_field_id = group_fields[0] if group_fields else None
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    display(grouped_df.head())
else:
    print('No categorical field found for grouping.')


## 5. Visualization
Visualize distribution of the selected numeric field and its normalized version.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)

plt.subplot(1, 2, 2)
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True)
plt.title(f"Distribution of Normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.tight_layout()
plt.show()


## 6. Conclusion
In this notebook, you:
- Loaded Croissant metadata for a real FAIR dataset
- Inspected record sets and field structure via their `@id`
- Loaded records into pandas DataFrames
- Filtered, normalized, and grouped numeric data for EDA
- Visualized distributions

You can now use the record set and field `@id` approach to extend this analysis for downstream ML or domain work. For more on Croissant, see [mlcroissant documentation](https://mlcommons.github.io/croissant/).